In [ ]:
# Parameters
input_image = None  # papermill will inject the uploaded image path


In [ ]:
import torch
import torchvision.transforms as T
import numpy as np
from PIL import Image
from pathlib import Path
import segmentation_models_pytorch as smp
import os


In [ ]:
# Model config (must match your training)
IN_CH = 1
N_CLASSES = 6
ENCODER = 'resnet50'
MODEL_PATH = 'models/best_model.pth'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:
# Load model
model = smp.Unet(encoder_name=ENCODER, encoder_weights=None, in_channels=IN_CH, classes=N_CLASSES)
state = torch.load(MODEL_PATH, map_location='cpu')
try:
    model.load_state_dict(state)
except Exception as e:
    # try if state contains 'state_dict' key
    if isinstance(state, dict) and 'state_dict' in state:
        model.load_state_dict(state['state_dict'])
    else:
        raise
model.to(device)
model.eval()
print('Model loaded from', MODEL_PATH)


In [ ]:
# Read input image
assert input_image is not None, 'input_image parameter not provided to notebook.'
inp = Path(input_image)
img = Image.open(str(inp)).convert('L')  # grayscale
transform = T.Compose([T.Resize((512,512)), T.ToTensor()])
x = transform(img).unsqueeze(0).to(device)

with torch.no_grad():
    pred = model(x)   # [B, C, H, W]
    probs = torch.softmax(pred, dim=1)
    mask = torch.argmax(probs, dim=1).squeeze(0).cpu().numpy().astype('uint8')

out_dir = Path('notebooks/output_oct')
out_dir.mkdir(parents=True, exist_ok=True)
mask_path = out_dir / f'mask_{inp.stem}.png'
from PIL import Image as PILImage
PILImage.fromarray(mask).save(mask_path)
print('Saved mask to', mask_path)
mask_path
